# Lección 10: Cálculo de Raíces, Algoritmos de Resolución y Aceleración
### Cálculo I y Fundamentos de la Física Matemática

---

Este cuaderno de Jupyter establece la **fundamentación formal de los métodos numéricos para el cálculo de raíces y la aceleración de convergencia**. Abordamos el método cerrado de bisección y el teorema de Bolzano, el método del punto fijo con su teorema de existencia y unicidad (contracción de Banach), y los métodos abiertos de la secante y Newton-Raphson. Asimismo, analizamos las modificaciones para raíces múltiples (método de Schröder y Von Mises). Concluimos con las técnicas de aceleración de convergencia lineal a cuadrática: la aceleración $\Delta^2$ de Aitken y el algoritmo de Steffensen, implementando simulaciones interactivas premium en Python.

---


## Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Aplicar** el método de bisección para resolver ecuaciones no lineales de forma segura, evaluando diferentes criterios de parada.
2. **Analizar** la convergencia y unicidad del método del punto fijo mediante la condición de contracción de la derivada.
3. **Utilizar** el método de la secante como una aproximación libre de derivadas de Newton-Raphson.
4. **Dominar** el método de Newton-Raphson, su interpretación geométrica y sus limitaciones cerca de raíces múltiples.
5. **Implementar** el método de Schröder (Newton modificado) para restaurar la convergencia cuadrática en raíces de alta multiplicidad.
6. **Acelerar** la convergencia lineal de procesos iterativos aplicando el método de la diferencia $\Delta^2$ de Aitken y el algoritmo de Steffensen.


## 1. El Método de Bisección (Bolzano)

### 1.1 Fundamento Teórico
El **método de bisección** (o de búsqueda binaria) es un método cerrado que se basa en el **Teorema del Valor Intermedio (Bolzano)**. 

> **Teorema**: Si una función $f(x)$ es continua en un intervalo cerrado $[a, b]$, y se cumple que $f(a) \cdot f(b) < 0$ (es decir, poseen signos opuestos), entonces existe al menos un valor real $p \in (a, b)$ tal que $f(p) = 0$.

El algoritmo divide repetidamente a la mitad el intervalo:
1. Definir los extremos iniciales $a_1 = a, b_1 = b$, y calcular el punto medio:
   $$p_1 = \frac{a_1 + b_1}{2}$$
2. Si $f(p_1) = 0$, la raíz es $p = p_1$ y el método finaliza.
3. Si $f(p_1) \neq 0$, analizamos el signo de los subintervalos:
   - Si $f(p_1) \cdot f(a_1) > 0$ (tienen el mismo signo), la raíz se encuentra en $[p_1, b_1] \implies a_2 = p_1, b_2 = b_1$.
   - Si $f(p_1) \cdot f(b_1) > 0$ (tienen el mismo signo), la raíz se encuentra en $[a_1, p_1] \implies a_2 = a_1, b_2 = p_1$.
4. Repetir el proceso para el nuevo intervalo $[a_2, b_2]$.

### 1.2 Criterios de Parada
El bucle iterativo se detiene cuando se satisface una de las siguientes condiciones de tolerancia (para $\epsilon > 0$):
1. **Tolerancia del Intervalo**: La longitud del intervalo es menor que la tolerancia:
   $$b_n - a_n < \epsilon \quad \text{o} \quad \frac{b_n - a_n}{2} < \epsilon$$
2. **Tolerancia de Aproximaciones**: La diferencia entre aproximaciones sucesivas es pequeña:
   $$|p_n - p_{n-1}| < \epsilon \quad \text{o} \quad \frac{|p_n - p_{n-1}|}{|p_n|} < \epsilon \quad (p_n \neq 0)$$
3. **Tolerancia de la Función**: El valor de la función en el punto medio es cercano a cero:
   $$|f(p_n)| < \epsilon$$
4. **Límite de Iteraciones**: Evita ciclos infinitos en caso de divergencia o errores de codificación estableciendo un número máximo de iteraciones $N_0$.

**Ejemplo**: Encontrar la raíz de $f(x) = 2x^3 - x^2 + x - 1$ en $[0, 1]$:
- $f(0) = -1 < 0$ y $f(1) = 1 > 0 \implies$ Cumple Bolzano.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configuración de estilo visual premium
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

def f(x):
    return 2.0*x**3 - x**2 + x - 1.0

# Bisección en [0, 1]
a, b = 0.0, 1.0
intervals = []
roots = []

for i in range(7):
    p = (a + b) / 2.0
    intervals.append((a, b))
    roots.append(p)
    if f(p) * f(a) > 0:
        a = p
    else:
        b = p

# Graficar reducción de intervalos
fig, axs = plt.subplots(1, 2, figsize=(14, 6))

# Subplot 1: Curva y raíz
x_vals = np.linspace(0, 1, 200)
axs[0].plot(x_vals, f(x_vals), 'b-', linewidth=2.5, label='$f(x) = 2x^3 - x^2 + x - 1$')
axs[0].axhline(0.0, color='black', linestyle='-')
axs[0].scatter(roots[-1], f(roots[-1]), color='red', s=100, zorder=5, label=f'Raíz aprox $\\approx {roots[-1]:.4f}$')
axs[0].set_title('Función y Raíz', fontweight='bold', color='navy')
axs[0].set_xlabel('$x$')
axs[0].set_ylabel('$f(x)$')
axs[0].legend()

# Subplot 2: Encogimiento de los intervalos
for idx, (a_i, b_i) in enumerate(intervals):
    y_pos = len(intervals) - idx
    axs[1].plot([a_i, b_i], [y_pos, y_pos], 'go-', linewidth=2, markersize=6)
    axs[1].text((a_i + b_i)/2.0, y_pos + 0.15, f'Iteración {idx+1}', ha='center', fontsize=9, fontweight='bold', color='darkgreen')
axs[1].axvline(roots[-1], color='red', linestyle='--', label='Línea de la Raíz')
axs[1].set_title('Encogimiento del Intervalo de Bisección', fontweight='bold', color='navy')
axs[1].set_xlabel('$x$')
axs[1].set_ylabel('Iteración')
axs[1].set_ylim(0.5, len(intervals) + 0.8)
axs[1].legend()

plt.tight_layout()
plt.show()


## 2. El Algoritmo del Punto Fijo

### 2.1 Definición y Teorema de Existencia y Unicidad
Un **punto fijo** de una función $g(x)$ es un número real $p$ tal que $p = g(p)$. El método del punto fijo transforma la ecuación $f(x) = 0$ en la forma equivalente $g(x) = x$ (ej. $g(x) = x - f(x)$).

> **Teorema de Existencia y Unicidad**: 
> 1. **Existencia**: Si $g \in C[a, b]$ y se cumple que $g(x) \in [a, b]$ para todo $x \in [a, b]$, entonces $g$ posee al menos un punto fijo en $[a, b]$.
> 2. **Unicidad**: Si además la derivada $g'(x)$ existe en $(a, b)$ y está acotada por una constante $k$ tal que:
>    $$|g'(x)| \le k < 1 \quad \forall x \in (a, b)$$
>    entonces el punto fijo $p$ en $[a, b]$ es **único**.

### 2.2 Demostración Formal de la Unicidad
Procedemos por contradicción suponiendo la existencia de dos puntos fijos distintos $p$ y $q$ en $[a, b]$ ($p = g(p)$, $q = g(q)$ con $p \neq q$).
- Por el Teorema del Valor Medio, dado que $g$ es continua en $[a, b]$ y derivable en $(a, b)$, existe un número $\xi$ estrictamente entre $p$ y $q$ tal que:
  $$g(p) - g(q) = g'(\xi) (p - q)$$
- Sustituyendo la propiedad de punto fijo:
  $$p - q = g'(\xi) (p - q)$$
- Aplicando valor absoluto a ambos lados y utilizando la cota de la derivada $|g'(x)| \le k < 1$:
  $$|p - q| = |g'(\xi)| |p - q| \le k |p - q| < |p - q|$$
- Hemos llegado a la desigualdad absurda:
  $$|p - q| < |p - q|$$
Esta contradicción proviene de suponer que $p \neq q$, por lo tanto, $p$ debe ser igual a $q$. Queda demostrada la unicidad del punto fijo. Q.E.D.

### 2.3 Ejemplo Resuelto de la Literatura
Sea $g(x) = \frac{x^2 - 1}{3}$ en el intervalo $[-1, 1]$.
- **Existencia**: La función es decreciente y creciente, con mínimo absoluto en $x=0 \implies g(0) = -1/3$ y máximo en $x=\pm 1 \implies g(\pm 1) = 0$. Dado que el rango de la función en el intervalo es $[-1/3, 0] \subseteq [-1, 1]$, la condición de existencia se cumple.
- **Unicidad**: La derivada es $g'(x) = \frac{2x}{3}$. En el intervalo $[-1, 1]$:
  $$|g'(x)| = \left| \frac{2x}{3} \right| \le \frac{2}{3} < 1 \quad \forall x \in [-1, 1]$$
  La constante de contracción es $k = 2/3 < 1$. Por lo tanto, el punto fijo es único.
- **Cálculo de la raíz exacta**:
  $$p = \frac{p^2 - 1}{3} \implies p^2 - 3p - 1 = 0 \implies p = \frac{3 - \sqrt{13}}{2} \approx -0.30277$$


In [ ]:
# Graficación de Telaraña (Cobweb Plot) para g(x) = (x^2 - 1)/3
def g(x):
    return (x**2 - 1.0) / 3.0

x_0 = 0.5
iters = [x_0]
curr_x = x_0
for _ in range(6):
    curr_x = g(curr_x)
    iters.append(curr_x)

x_grid = np.linspace(-1.1, 1.1, 300)

plt.figure(figsize=(9, 7))
plt.plot(x_grid, x_grid, 'k-', alpha=0.5, label='$y = x$')
plt.plot(x_grid, g(x_grid), 'b-', linewidth=2.0, label='$y = g(x) = \\frac{x^2 - 1}{3}$')

# Trazar camino de la telaraña
x_path, y_path = [], []
x_path.append(iters[0])
y_path.append(0.0)

for idx in range(len(iters) - 1):
    x_path.append(iters[idx])
    y_path.append(iters[idx+1])
    x_path.append(iters[idx+1])
    y_path.append(iters[idx+1])

plt.plot(x_path, y_path, 'r--', linewidth=1.5, label='Camino de iteración (Cobweb)')
plt.scatter(iters, [g(x) for x in iters], color='red', s=40, zorder=5)

p_exact = (3.0 - np.sqrt(13.0)) / 2.0
plt.scatter(p_exact, p_exact, color='darkgreen', s=120, marker='*', zorder=6, label=f'Punto Fijo $p \\approx {p_exact:.5f}$')

plt.title('Diagrama de Telaraña de la Iteración de Punto Fijo', fontweight='bold', fontsize=13, color='navy')
plt.xlabel('$x$')
plt.ylabel('$y$')
plt.xlim(-1.1, 1.1)
plt.ylim(-0.8, 1.1)
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


## 3. El Método de la Secante y Newton-Raphson

### 3.1 Método de Newton-Raphson
Es el método numérico abierto más utilizado para encontrar raíces. Se deriva linealizando la función mediante su polinomio de Taylor de primer grado alrededor de la aproximación actual $x_n$:
$$f(x) \approx f(x_n) + f'(x_n)(x - x_n)$$
Igualando a cero para encontrar el cruce del eje $x$ ($f(x_{n+1}) = 0$):
$$0 = f(x_n) + f'(x_n)(x_{n+1} - x_n) \implies x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$

### 3.2 Método de la Secante
El inconveniente de Newton-Raphson es la necesidad de evaluar la derivada $f'(x_n)$ en cada paso, lo cual puede ser muy costoso computacionalmente o inviable si no se conoce la forma analítica de la función. 

El **método de la secante** resuelve esto reemplazando la derivada por una aproximación de diferencia finita hacia atrás basada en las últimas dos iteraciones:
$$f'(x_n) \approx \frac{f(x_n) - f(x_{n-1})}{x_n - x_{n-1}}$$

Sustituyendo esto en la fórmula de Newton-Raphson se obtiene la fórmula recursiva (que requiere dos estimaciones iniciales $x_0, x_1$):
$$x_{n+1} = x_n - f(x_n) \frac{x_n - x_{n-1}}{f(x_n) - f(x_{n-1})}$$

### 3.3 Resolución del Ejemplo de la Literatura (Corrección de Errata)
**Ejemplo**: Encontrar la intersección de $f(x) = \sin(x/2)$ y $g(x) = 5e^{-x}$.
- Ecuación asociada: $F(x) = \sin(x/2) - 5e^{-x} = 0$.
- Valores de partida: $x_0 = 0$, $x_1 = 1$.
  - $F(x_0) = F(0) = \sin(0) - 5e^0 = -5.0$
  - $F(x_1) = F(1) = \sin(0.5) - 5e^{-1} \approx 0.4794255 - 1.8393972 = -1.3599717$

> **Nota de Calidad (Errata en el texto original)**: El texto del PDF original contiene un error tipográfico en la fórmula del cálculo de $x_2$ al listar el valor de $F(x_1)$ como `-1.8393` (que corresponde únicamente al término $-5e^{-1}$ sin sumarle la parte del seno). Sin embargo, el resultado de la iteración listado en el texto original como $x_2 = 1.3736$ fue calculado utilizando el valor **correcto** de $F(x_1) \approx -1.35997$. Comprobamos analíticamente:
> $$x_2 = 1 - (-1.35997) \cdot \frac{1 - 0}{-1.35997 - (-5)} = 1 + \frac{1.35997}{3.64003} = 1 + 0.373615 = 1.3736$$
> lo cual confirma la exactitud matemática de nuestra corrección.

Las iteraciones sucesivas convergen rápidamente a la raíz:
- $x_3 = 1.6978$
- $x_4 = 1.8122$
- $x_5 = 1.8369$
- $x_6 = 1.8386$ (donde $F(x_6) \approx 0 \implies f(x_6) = g(x_6) \approx 0.7951$).


In [ ]:
# Simulación del Método de la Secante para F(x) = sin(x/2) - 5e^-x
def F(x):
    return np.sin(x / 2.0) - 5.0 * np.exp(-x)

x_0, x_1 = 0.0, 1.0
sec_points = [x_0, x_1]

print("Iteraciones del Método de la Secante:")
print(f"x0 = {x_0:.4f} | F(x0) = {F(x_0):.6f}")
print(f"x1 = {x_1:.4f} | F(x1) = {F(x_1):.6f}")

for i in range(2, 7):
    x_prev = sec_points[-1]
    x_pprev = sec_points[-2]
    # Fórmula de la Secante
    x_new = x_prev - F(x_prev) * (x_prev - x_pprev) / (F(x_prev) - F(x_pprev))
    sec_points.append(x_new)
    print(f"x{i} = {x_new:.4f} | F(x{i}) = {F(x_new):.6f}")

# Graficar curvas de intersección
x_grid = np.linspace(-0.2, 2.5, 300)

plt.figure(figsize=(10, 6.5))
plt.plot(x_grid, np.sin(x_grid/2.0), 'g-', linewidth=2.0, label='$f(x) = \\sin(x/2)$')
plt.plot(x_grid, 5.0*np.exp(-x_grid), 'r-', linewidth=2.0, label='$g(x) = 5e^{-x}$')
plt.axvline(sec_points[-1], color='purple', linestyle='--', label=f'Cruce $x \\approx {sec_points[-1]:.4f}$')
plt.scatter(sec_points[-1], np.sin(sec_points[-1]/2.0), color='black', s=80, zorder=5)

plt.title('Intersección de Curvas: Método de la Secante', fontweight='bold')
plt.xlabel('$x$')
plt.ylabel('$y$')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


## 4. Métodos de Newton Modificados (Raíces Múltiples)

### 4.1 Pérdida de Convergencia Cuadrática
Cuando una raíz $p$ tiene una **multiplicidad $m > 1$** (es decir, $f(p) = f'(p) = \dots = f^{(m-1)}(p) = 0$ y $f^{(m)}(p) \neq 0$), el método clásico de Newton-Raphson pierde su velocidad de convergencia cuadrática y pasa a converger de forma puramente lineal, siendo muy lento.

### 4.2 Métodos de Aceleración y Schröder
Existen dos alternativas clásicas para restaurar la convergencia cuadrática:
1. **Conociendo la multiplicidad $m$**: Se incorpora $m$ como factor en la corrección:
   $$x_{n+1} = x_n - m \frac{f(x_n)}{f'(x_n)}$$
2. **Sin conocer la multiplicidad (Método de Schröder)**: Se define la función auxiliar:
   $$u(x) = \frac{f(x)}{f'(x)}$$
   Las raíces de $u(x)$ coinciden exactamente con las de $f(x)$, pero todas tienen **multiplicidad igual a 1** (raíces simples). Aplicando Newton-Raphson a $u(x)$:
   $$x_{n+1} = x_n - \frac{u(x_n)}{u'(x_n)}$$
   Derivando $u(x)$ mediante la regla del cociente:
   $$u'(x) = \frac{(f'(x))^2 - f(x)f''(x)}{(f'(x))^2}$$
   Sustituyendo en la recursión se obtiene la fórmula modificada de Schröder:
   $$x_{n+1} = x_n - \frac{f(x_n) f'(x_n)}{(f'(x_n))^2 - f(x_n) f''(x_n)}$$
3. **Fórmula de Von Mises**: Si la derivada varía poco en el intervalo, se puede fijar la derivada en el punto inicial $p_0$ para ahorrar el cálculo de derivadas en cada iteración:
   $$p_{n+1} = p_n - \frac{f(p_n)}{f'(p_0)}$$


In [ ]:
# Comparación de Convergencia: Newton Clásico vs Schröder
# Función con raíz múltiple: f(x) = (x - 1.5)^2 * ln(x)
# Raíz alpha = 1.5 de multiplicidad m = 2
import sympy as sp

x_sym = sp.Symbol('x')
f_sym = (x_sym - 1.5)**2 * sp.log(x_sym)
df_sym = sp.diff(f_sym, x_sym)
ddf_sym = sp.diff(df_sym, x_sym)

# Convertimos a funciones ejecutables de numpy
f_func = sp.lambdify(x_sym, f_sym, 'numpy')
df_func = sp.lambdify(x_sym, df_sym, 'numpy')
ddf_func = sp.lambdify(x_sym, ddf_sym, 'numpy')

x_0 = 1.8
alpha = 1.5

# Newton Clásico
newton_errors = []
curr_x = x_0
for _ in range(8):
    newton_errors.append(abs(curr_x - alpha))
    curr_x = curr_x - f_func(curr_x)/df_func(curr_x)

# Schröder Modificado
schroder_errors = []
curr_x = x_0
for _ in range(8):
    schroder_errors.append(abs(curr_x - alpha))
    # Evitamos división por cero si estamos muy cerca de la raíz
    denom = (df_func(curr_x))**2 - f_func(curr_x)*ddf_func(curr_x)
    if abs(denom) < 1e-15:
        break
    curr_x = curr_x - (f_func(curr_x)*df_func(curr_x)) / denom

# Graficar errores en escala logarítmica para verificar velocidad
plt.figure(figsize=(9, 6))
plt.semilogy(newton_errors, 'ro-', linewidth=2.0, label='Newton Clásico (Convergencia Lineal)')
plt.semilogy(schroder_errors, 'gs-', linewidth=2.0, label='Método de Schröder (Convergencia Cuadrática)')
plt.title('Restauración de Convergencia Cuadrática en Raíz Múltiple ($m=2$)', fontweight='bold')
plt.xlabel('Iteración ($n$)')
plt.ylabel('Error absoluto $|x_n - 1.5|$ (Escala Logarítmica)')
plt.legend()
plt.grid(True, which="both", linestyle='--', alpha=0.6)
plt.show()


## 5. Técnicas de Aceleración: Delta-2 de Aitken y Steffensen

### 5.1 Aceleración $\Delta^2$ de Aitken
Se utiliza para acelerar cualquier sucesión $\{p_n\}_{n=0}^\infty$ que converja linealmente a un límite $p$. Se asume que para un índice $n$ suficientemente grande:
$$\frac{p_{n+1} - p}{p_n - p} \approx \frac{p_{n+2} - p}{p_{n+1} - p} \approx \lambda \quad (|\lambda| < 1)$$
Multiplicando de forma cruzada para despejar el valor del límite $p$:
$$(p_{n+1} - p)^2 \approx (p_{n+2} - p)(p_n - p)$$
$$p_{n+1}^2 - 2p_{n+1} p + p^2 \approx p_n p_{n+2} - (p_n + p_{n+2}) p + p^2$$
$$p(p_{n+2} - 2p_{n+1} + p_n) \approx p_n p_{n+2} - p_{n+1}^2$$
$$p \approx \frac{p_n p_{n+2} - p_{n+1}^2}{p_{n+2} - 2p_{n+1} + p_n}$$

Reescribiendo algebraicamente para aislar $p_n$ en el numerador:
$$\hat{p}_n = p_n - \frac{(p_{n+1} - p_n)^2}{p_{n+2} - 2p_{n+1} + p_n}$$

Utilizando la notación de diferencias progresivas $\Delta p_n = p_{n+1} - p_n$ y $\Delta^2 p_n = p_{n+2} - 2p_{n+1} + p_n$:
$$\hat{p}_n = p_n - \frac{(\Delta p_n)^2}{\Delta^2 p_n}$$
La Sucesión acelerada $\{\hat{p}_n\}$ converge significativamente más rápido al límite $p$.

### 5.2 Algoritmo de Steffensen
El **método de Steffensen** aplica la aceleración de Aitken de forma recursiva a una iteración de punto fijo ordinaria $p_{n+1} = g(p_n)$:
- Partiendo de $p_0^{(k)}$.
- Calculamos dos pasos de punto fijo ordinario:
  $$p_1^{(k)} = g(p_0^{(k)}), \quad p_2^{(k)} = g(p_1^{(k)})$$
- Aplicamos la fórmula de Aitken para calcular el valor acelerado, que se convierte en el nuevo punto inicial para el siguiente paso recursivo:
  $$p_0^{(k+1)} = p_0^{(k)} - \frac{(p_1^{(k)} - p_0^{(k)})^2}{p_2^{(k)} - 2p_1^{(k)} + p_0^{(k)}}$$

Este método acelera la convergencia lineal a **convergencia cuadrática** ($O(e_n^2)$) sin evaluar derivadas, lo cual lo hace extremadamente potente y eficiente.

In [ ]:
# 1. Simulación de la Sucesión Acelerada de Aitken para p_n = cos(1/n) -> Límite = 1
n_vals = np.arange(1, 20)
p_seq = np.cos(1.0 / n_vals)

# Aplicamos Aitken a la sucesión
p_hat = []
for i in range(len(p_seq) - 2):
    num_ait = (p_seq[i+1] - p_seq[i])**2
    denom_ait = p_seq[i+2] - 2.0*p_seq[i+1] + p_seq[i]
    p_hat.append(p_seq[i] - num_ait / denom_ait)

fig, axs = plt.subplots(1, 2, figsize=(14, 6))

axs[0].plot(n_vals, np.abs(p_seq - 1.0), 'bo-', label='Sucesión Original $p_n$')
axs[0].plot(n_vals[:-2], np.abs(np.array(p_hat) - 1.0), 'r^-', label='Aceleración de Aitken $\\hat{p}_n$')
axs[0].set_yscale('log')
axs[0].set_title('Aceleración de Aitken para $p_n = \\cos(1/n)$', fontweight='bold', color='navy')
axs[0].set_xlabel('$n$')
axs[0].set_ylabel('Error absoluto $|p_n - 1|$ (Escala Logarítmica)')
axs[0].legend()

# 2. Método de Steffensen para resolver x = cos(x) (Dottie number)
def g_dot(x):
    return np.cos(x)

x_start = 0.5
dottie_root = 0.7390851332151607

# Punto Fijo clásico
pf_err = []
curr = x_start
for _ in range(8):
    pf_err.append(abs(curr - dottie_root))
    curr = g_dot(curr)

# Steffensen
steff_err = []
curr = x_start
for _ in range(5):
    steff_err.append(abs(curr - dottie_root))
    p1 = g_dot(curr)
    p2 = g_dot(p1)
    denom = p2 - 2.0*p1 + curr
    if abs(denom) < 1e-15:
        break
    curr = curr - (p1 - curr)**2 / denom

axs[1].plot(pf_err, 'bo-', label='Punto Fijo Clásico')
axs[1].plot(steff_err, 'g^-', label='Método de Steffensen')
axs[1].set_yscale('log')
axs[1].set_title('Aceleración de Steffensen para $x = \\cos(x)$', fontweight='bold', color='navy')
axs[1].set_xlabel('Iteración')
axs[1].set_ylabel('Error absoluto (Escala Logarítmica)')
axs[1].legend()

plt.tight_layout()
plt.show()


## 6. Resumen de la Lección

1. **Bisección**: Método cerrado seguro que reduce a la mitad el intervalo basándose en Bolzano. Posee convergencia lineal garantizada.
2. **Punto Fijo**: Permite hallar raíces resolviendo $g(x)=x$. Converge de forma única en $[a,b]$ si la derivada satisface $|g'(x)| \le k < 1$.
3. **Secante**: Método abierto que aproxima la derivada mediante la secante de los últimos dos puntos, eliminando la necesidad del cálculo analítico de derivadas.
4. **Newton-Raphson**: Posee velocidad de convergencia cuadrática, pero requiere derivadas y pierde eficiencia en raíces múltiples.
5. **Newton Modificados**: El método de Schröder restaura la convergencia cuadrática en raíces de alta multiplicidad analizando $u(x) = f(x)/f'(x)$. La fórmula de Von Mises ahorra evaluaciones de derivadas fijándola en $p_0$.
6. **Aceleración**: El método $\Delta^2$ de Aitken acelera sucesiones de convergencia lineal. El algoritmo de Steffensen combina Aitken de forma recursiva con punto fijo logrando convergencia cuadrática sin derivadas.

## 7. Referencias Bibliográficas

1. Muto, A. (2022). *Solución aproximada de ecuaciones de una variable: preliminares*. Curso de métodos numéricos, Capítulo 5. [En línea]. Disponible en: http://www.ehu.eus/~mepmufov/html/Parte2.pdf
2. Muto, A. (2022). *Solución aproximada de ecuaciones de una variable: preliminares*. Curso de métodos numéricos, Capítulo 7. [En línea]. Disponible en: https://ocw.ehu.eus/file.php/81/capitulo-7.pdf
3. Russo, R. (2022). *Métodos numéricos: Apuntes y ejemplos*. Cátedra de métodos numéricos, Departamento de Matemática, Facultad de Ingeniería, Universidad Nacional de Mar del Plata. [En línea]. Disponible en: http://www3.fi.mdp.edu.ar/metodos/apuntes/secante_rodrigo.pdf
4. Vargas, J. (2018). *Método de Newton Raphson*. Facultad de ingeniería, Universidad Tecnológica de Bolívar. [En línea]. Disponible en: http://personal.cimat.mx:8181/~julio/courses/progra01/clase18/MetododeNewtonRaphson.pdf
5. Rodríguez, L. (2017). *Métodos numéricos para la aproximación de raíces múltiples de ecuaciones no lineales*. Departamento de matemática aplicada, Tesis de grado, Ingeniería Mecánica. [En línea]. Disponible en: https://1bestlinks.net/mRGyt
